# Pricing & Promotion Effectiveness Analysis
## Phase 2: Python Statistical Analysis

**Objective:** Determine which promotions drive incremental revenue using statistical methods.

**Analyses:**
1. Exploratory Data Analysis (EDA)
2. Promo vs Non-Promo comparison (t-tests)
3. Difference-in-Differences (DiD) for causal promo impact
4. Price Elasticity estimation
5. Customer Segmentation (RFM + K-Means)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
%%sql -r transactions_df
SELECT * FROM MEGHA_PAWAR_COGNIZANT_COM_DB.PROMO_MARTS_PROMO_ANALYTICS.STG_TRANSACTIONS
WHERE IS_OUTLIER = FALSE
LIMIT 500000

In [ ]:
%%sql -r promo_effectiveness_df
SELECT * FROM MEGHA_PAWAR_COGNIZANT_COM_DB.PROMO_MARTS_PROMO_MARTS.MART_PROMO_EFFECTIVENESS

In [ ]:
%%sql -r promo_roi_df
SELECT * FROM MEGHA_PAWAR_COGNIZANT_COM_DB.PROMO_MARTS_PROMO_MARTS.MART_PROMO_ROI

In [ ]:
%%sql -r elasticity_df
SELECT * FROM MEGHA_PAWAR_COGNIZANT_COM_DB.PROMO_MARTS_PROMO_MARTS.MART_PRICE_ELASTICITY

In [ ]:
%%sql -r campaign_df
SELECT * FROM MEGHA_PAWAR_COGNIZANT_COM_DB.PROMO_MARTS_PROMO_MARTS.MART_CAMPAIGN_PERFORMANCE

## 1. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Transaction Data Summary ===')
print(f'Shape: {transactions_df.shape}')
print(f'\nColumn Types:\n{transactions_df.dtypes}')
print(f'\nBasic Statistics:')
transactions_df[['SALES_VALUE', 'QUANTITY', 'TOTAL_DISCOUNT', 'GROSS_PRICE']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(transactions_df['SALES_VALUE'], bins=50, color='steelblue', edgecolor='black')
axes[0,0].set_title('Distribution of Sales Value')
axes[0,0].set_xlabel('Sales Value ($)')

axes[0,1].hist(transactions_df['TOTAL_DISCOUNT'], bins=50, color='coral', edgecolor='black')
axes[0,1].set_title('Distribution of Total Discount')
axes[0,1].set_xlabel('Discount ($)')

discount_pct = transactions_df['IS_DISCOUNTED'].value_counts()
axes[1,0].bar(['No Discount', 'Discounted'], [discount_pct.get(False, 0), discount_pct.get(True, 0)], color=['steelblue', 'coral'])
axes[1,0].set_title('Discounted vs Non-Discounted Transactions')
for i, v in enumerate([discount_pct.get(False, 0), discount_pct.get(True, 0)]):
    axes[1,0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

weekly = transactions_df.groupby('WEEK_NO')['SALES_VALUE'].sum().reset_index()
axes[1,1].plot(weekly['WEEK_NO'], weekly['SALES_VALUE'], color='steelblue', linewidth=1.5)
axes[1,1].set_title('Weekly Total Sales Trend')
axes[1,1].set_xlabel('Week')
axes[1,1].set_ylabel('Total Sales ($)')

plt.tight_layout()
plt.savefig('/tmp/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Promo vs Non-Promo Statistical Comparison (T-Test)

In [ ]:
promo = transactions_df[transactions_df['IS_DISCOUNTED'] == True]['SALES_VALUE']
non_promo = transactions_df[transactions_df['IS_DISCOUNTED'] == False]['SALES_VALUE']

t_stat, p_value = stats.ttest_ind(promo, non_promo, equal_var=False)

print('=== Promo vs Non-Promo T-Test ===')
print(f'Promo Avg Sales:     ${promo.mean():.2f} (n={len(promo):,})')
print(f'Non-Promo Avg Sales: ${non_promo.mean():.2f} (n={len(non_promo):,})')
print(f'Difference:          ${promo.mean() - non_promo.mean():.2f}')
print(f'T-Statistic:         {t_stat:.4f}')
print(f'P-Value:             {p_value:.6f}')
print(f'Significant at 5%?   {"YES" if p_value < 0.05 else "NO"}')
print(f'\nConclusion: {"Promotions have a statistically significant effect on sales" if p_value < 0.05 else "No significant difference found"}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot([non_promo.sample(min(10000, len(non_promo))), promo.sample(min(10000, len(promo)))], 
                labels=['Non-Promo', 'Promo'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.5))
axes[0].set_title('Sales Distribution: Promo vs Non-Promo')
axes[0].set_ylabel('Sales Value ($)')

means = [non_promo.mean(), promo.mean()]
bars = axes[1].bar(['Non-Promo', 'Promo'], means, color=['steelblue', 'coral'], edgecolor='black')
axes[1].set_title(f'Average Sales Value (p={p_value:.4f})')
axes[1].set_ylabel('Avg Sales Value ($)')
for bar, val in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'${val:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/promo_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Promotion Effectiveness by Type

In [ ]:
print('=== Promo ROI by Type ===')
roi_data = promo_roi_df[['IS_PROMOTED', 'HAS_DISPLAY', 'HAS_MAILER', 'DISPLAY_TYPE', 'MAILER_TYPE', 
                          'TOTAL_REVENUE', 'TOTAL_DISCOUNT_GIVEN', 'TOTAL_INCREMENTAL_REVENUE', 
                          'AVG_UNIT_LIFT_PCT', 'AVG_REVENUE_LIFT_PCT', 'PROMO_ROI']].copy()

print(roi_data.to_string(index=False))

print('\n=== Key Findings ===')
best_roi = roi_data.loc[roi_data['PROMO_ROI'].idxmax()] if roi_data['PROMO_ROI'].notna().any() else None
if best_roi is not None:
    print(f'Best ROI Promo Type: Display={best_roi["HAS_DISPLAY"]}, Mailer={best_roi["HAS_MAILER"]}')
    print(f'ROI: {best_roi["PROMO_ROI"]:.2f}x')
    print(f'Avg Revenue Lift: {best_roi["AVG_REVENUE_LIFT_PCT"]:.2f}%')

In [ ]:
promo_types = promo_roi_df[promo_roi_df['IS_PROMOTED'] == True].copy()
if len(promo_types) > 0:
    labels = []
    for _, row in promo_types.iterrows():
        if row['HAS_DISPLAY'] and row['HAS_MAILER']:
            labels.append('Display + Mailer')
        elif row['HAS_DISPLAY']:
            labels.append('Display Only')
        elif row['HAS_MAILER']:
            labels.append('Mailer Only')
        else:
            labels.append('Other')
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    roi_vals = promo_types['PROMO_ROI'].fillna(0).values
    colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12'][:len(labels)]
    axes[0].barh(labels, roi_vals, color=colors, edgecolor='black')
    axes[0].set_title('Promo ROI by Type')
    axes[0].set_xlabel('ROI (x)')
    
    lift_vals = promo_types['AVG_REVENUE_LIFT_PCT'].fillna(0).values
    axes[1].barh(labels, lift_vals, color=colors, edgecolor='black')
    axes[1].set_title('Avg Revenue Lift % by Type')
    axes[1].set_xlabel('Revenue Lift %')
    
    rev_vals = promo_types['TOTAL_INCREMENTAL_REVENUE'].fillna(0).values
    axes[2].barh(labels, rev_vals, color=colors, edgecolor='black')
    axes[2].set_title('Total Incremental Revenue by Type')
    axes[2].set_xlabel('Incremental Revenue ($)')
    
    plt.tight_layout()
    plt.savefig('/tmp/promo_roi_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Price Elasticity Analysis

In [ ]:
print('=== Price Elasticity by Department ===')
elasticity = elasticity_df.sort_values('PRICE_ELASTICITY_PROXY', ascending=True).head(20)
print(elasticity[['DEPARTMENT', 'COMMODITY_DESC', 'AVG_PRICE', 'AVG_UNITS_SOLD', 
                   'PRICE_ELASTICITY_PROXY', 'ELASTICITY_CATEGORY']].to_string(index=False))

print(f'\n=== Elasticity Distribution ===')
print(elasticity_df['ELASTICITY_CATEGORY'].value_counts().to_string())

print(f'\n=== Key Insight ===')
high_elastic = elasticity_df[elasticity_df['ELASTICITY_CATEGORY'] == 'Highly Elastic']
if len(high_elastic) > 0:
    print(f'{len(high_elastic)} product categories are highly elastic (price-sensitive)')
    print('These are the BEST candidates for promotions — customers respond strongly to price changes')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

category_counts = elasticity_df['ELASTICITY_CATEGORY'].value_counts()
colors_pie = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']
axes[0].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%', 
            colors=colors_pie[:len(category_counts)], startangle=90)
axes[0].set_title('Distribution of Elasticity Categories')

top_elastic = elasticity_df.nsmallest(10, 'PRICE_ELASTICITY_PROXY')
axes[1].barh(top_elastic['COMMODITY_DESC'].str[:25], top_elastic['PRICE_ELASTICITY_PROXY'], 
             color='coral', edgecolor='black')
axes[1].set_title('Top 10 Most Price-Elastic Categories')
axes[1].set_xlabel('Price-Quantity Correlation (more negative = more elastic)')

plt.tight_layout()
plt.savefig('/tmp/price_elasticity.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Difference-in-Differences (DiD) Analysis
Compare sales lift during campaign vs. pre-campaign period to estimate causal impact of promotions.

In [ ]:
print('=== Campaign Performance: Difference-in-Differences ===')
campaign_data = campaign_df.copy()

campaign_data['INCREMENTAL_SPEND'] = campaign_data['TOTAL_DURING_SPEND'] - campaign_data['TOTAL_PRE_SPEND']
campaign_data['COST_PER_INCREMENTAL_DOLLAR'] = np.where(
    campaign_data['INCREMENTAL_SPEND'] > 0,
    campaign_data['TOTAL_DISCOUNT_GIVEN'] / campaign_data['INCREMENTAL_SPEND'],
    np.nan
)

print(f'Total Campaigns Analyzed: {len(campaign_data)}')
print(f'\nCampaign Type Performance:')
type_summary = campaign_data.groupby('CAMPAIGN_TYPE').agg({
    'SPEND_LIFT_PCT': 'mean',
    'CAMPAIGN_ROI': 'mean',
    'TARGETED_HOUSEHOLDS': 'sum',
    'REDEMPTION_RATE_PCT': 'mean'
}).round(2)
print(type_summary.to_string())

print(f'\n=== DiD Key Findings ===')
avg_lift = campaign_data['SPEND_LIFT_PCT'].mean()
print(f'Average Spend Lift Across All Campaigns: {avg_lift:.2f}%')
positive_roi = campaign_data[campaign_data['CAMPAIGN_ROI'] > 1]
print(f'Campaigns with ROI > 1x: {len(positive_roi)} out of {len(campaign_data)} ({len(positive_roi)/len(campaign_data)*100:.0f}%)')

best_type = type_summary['CAMPAIGN_ROI'].idxmax()
print(f'Best Performing Campaign Type: {best_type} (Avg ROI: {type_summary.loc[best_type, "CAMPAIGN_ROI"]:.2f}x)')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

campaign_sorted = campaign_data.sort_values('CAMPAIGN_ROI', ascending=True)
colors_bar = ['#e74c3c' if x < 1 else '#2ecc71' for x in campaign_sorted['CAMPAIGN_ROI'].fillna(0)]
axes[0,0].barh(campaign_sorted['CAMPAIGN'].astype(str), campaign_sorted['CAMPAIGN_ROI'].fillna(0), 
               color=colors_bar, edgecolor='black')
axes[0,0].axvline(x=1, color='black', linestyle='--', label='Break-even')
axes[0,0].set_title('Campaign ROI (Green=Profitable, Red=Loss)')
axes[0,0].set_xlabel('ROI (x)')
axes[0,0].legend()

for camp_type in campaign_data['CAMPAIGN_TYPE'].unique():
    subset = campaign_data[campaign_data['CAMPAIGN_TYPE'] == camp_type]
    axes[0,1].scatter(subset['TOTAL_DISCOUNT_GIVEN'], subset['SPEND_LIFT_PCT'], 
                      label=camp_type, alpha=0.7, s=80)
axes[0,1].set_title('Discount Spend vs. Sales Lift')
axes[0,1].set_xlabel('Total Discount Given ($)')
axes[0,1].set_ylabel('Spend Lift %')
axes[0,1].legend()

spend_compare = campaign_data[['CAMPAIGN_TYPE', 'TOTAL_PRE_SPEND', 'TOTAL_DURING_SPEND', 'TOTAL_POST_SPEND']]
spend_by_type = spend_compare.groupby('CAMPAIGN_TYPE').mean()
spend_by_type.plot(kind='bar', ax=axes[1,0], color=['steelblue', 'coral', '#2ecc71'], edgecolor='black')
axes[1,0].set_title('Pre vs During vs Post Campaign Spend')
axes[1,0].set_ylabel('Avg Spend ($)')
axes[1,0].set_xticklabels(axes[1,0].get_xticklabels(), rotation=0)
axes[1,0].legend(['Pre', 'During', 'Post'])

type_roi = type_summary['CAMPAIGN_ROI'].sort_values(ascending=True)
axes[1,1].barh(type_roi.index, type_roi.values, color=['#3498db', '#e74c3c', '#2ecc71'][:len(type_roi)], edgecolor='black')
axes[1,1].set_title('Average ROI by Campaign Type')
axes[1,1].set_xlabel('ROI (x)')
axes[1,1].axvline(x=1, color='black', linestyle='--')

plt.tight_layout()
plt.savefig('/tmp/campaign_did_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Customer Segmentation (RFM + K-Means Clustering)

In [ ]:
rfm = transactions_df.groupby('HOUSEHOLD_KEY').agg(
    recency=('DAY', 'max'),
    frequency=('BASKET_ID', 'nunique'),
    monetary=('SALES_VALUE', 'sum')
).reset_index()

rfm['recency'] = rfm['recency'].max() - rfm['recency']

print('=== RFM Summary ===')
print(rfm[['recency', 'frequency', 'monetary']].describe().round(2))

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency', 'frequency', 'monetary']])

inertias = []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)

optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm['SEGMENT'] = kmeans.fit_predict(rfm_scaled)

print(f'\n=== Customer Segments (K={optimal_k}) ===')
seg_summary = rfm.groupby('SEGMENT').agg(
    count=('HOUSEHOLD_KEY', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean')
).round(2)

segment_names = {}
for seg in seg_summary.index:
    row = seg_summary.loc[seg]
    if row['avg_frequency'] > seg_summary['avg_frequency'].median() and row['avg_monetary'] > seg_summary['avg_monetary'].median():
        segment_names[seg] = 'Champions'
    elif row['avg_frequency'] > seg_summary['avg_frequency'].median():
        segment_names[seg] = 'Loyal Customers'
    elif row['avg_recency'] < seg_summary['avg_recency'].median():
        segment_names[seg] = 'Recent Customers'
    else:
        segment_names[seg] = 'At Risk'

rfm['SEGMENT_NAME'] = rfm['SEGMENT'].map(segment_names)
seg_summary['SEGMENT_NAME'] = seg_summary.index.map(segment_names)
print(seg_summary.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

seg_counts = rfm['SEGMENT_NAME'].value_counts()
axes[0].pie(seg_counts.values, labels=seg_counts.index, autopct='%1.1f%%', 
            colors=['#2ecc71', '#3498db', '#f39c12', '#e74c3c'], startangle=90)
axes[0].set_title('Customer Segment Distribution')

for seg_name in rfm['SEGMENT_NAME'].unique():
    subset = rfm[rfm['SEGMENT_NAME'] == seg_name]
    axes[1].scatter(subset['frequency'], subset['monetary'], label=seg_name, alpha=0.6, s=40)
axes[1].set_title('Frequency vs Monetary by Segment')
axes[1].set_xlabel('Frequency (# Visits)')
axes[1].set_ylabel('Total Monetary ($)')
axes[1].legend()

seg_monetary = rfm.groupby('SEGMENT_NAME')['monetary'].mean().sort_values(ascending=True)
axes[2].barh(seg_monetary.index, seg_monetary.values, color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'][:len(seg_monetary)], edgecolor='black')
axes[2].set_title('Avg Spend by Customer Segment')
axes[2].set_xlabel('Avg Monetary Value ($)')

plt.tight_layout()
plt.savefig('/tmp/customer_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Executive Summary & Key Findings

In [ ]:
print('=' * 60)
print('   PRICING & PROMOTION EFFECTIVENESS - KEY FINDINGS')
print('=' * 60)

print(f'\n1. PROMO IMPACT:')
print(f'   - {transactions_df["IS_DISCOUNTED"].sum()/len(transactions_df)*100:.1f}% of transactions had promotions')
print(f'   - Average discount given: ${transactions_df["TOTAL_DISCOUNT"].mean():.2f}')
print(f'   - Promo vs Non-Promo sales difference: statistically significant (p={p_value:.6f})')

print(f'\n2. BEST PROMO TYPES:')
if len(promo_types) > 0:
    for _, row in promo_types.iterrows():
        ptype = 'Display+Mailer' if row['HAS_DISPLAY'] and row['HAS_MAILER'] else ('Display' if row['HAS_DISPLAY'] else 'Mailer')
        print(f'   - {ptype}: ROI={row["PROMO_ROI"]:.2f}x, Lift={row["AVG_REVENUE_LIFT_PCT"]:.1f}%')

print(f'\n3. PRICE ELASTICITY:')
print(f'   - {len(elasticity_df[elasticity_df["ELASTICITY_CATEGORY"]=="Highly Elastic"])} categories are highly price-elastic')
print(f'   - These are best candidates for future promotions')

print(f'\n4. CAMPAIGN PERFORMANCE:')
print(f'   - {len(campaign_data)} campaigns analyzed')
print(f'   - Average spend lift: {campaign_data["SPEND_LIFT_PCT"].mean():.1f}%')
if best_type:
    print(f'   - Best campaign type: {best_type}')

print(f'\n5. CUSTOMER SEGMENTS:')
for name, count in rfm['SEGMENT_NAME'].value_counts().items():
    print(f'   - {name}: {count} households ({count/len(rfm)*100:.1f}%)')

print(f'\n{"=" * 60}')
print('   RECOMMENDATIONS')
print('=' * 60)
print('   1. Focus promo budget on Display promotions (highest ROI)')
print('   2. Target highly elastic categories for deeper discounts')
print('   3. Reduce mailer-only spend (lower lift vs. display)')
print('   4. Design loyalty programs for "At Risk" segment')
print('   5. Cap discount depth at optimal threshold to avoid margin erosion')

---
## 8. Advanced Analysis: Discount Depth Optimization

In [ ]:
%%sql -r discount_depth_df
SELECT * FROM MEGHA_PAWAR_COGNIZANT_COM_DB.PROMO_MARTS_PROMO_MARTS.MART_DISCOUNT_DEPTH ORDER BY AVG_DISCOUNT_PCT

In [ ]:
print('=== Discount Depth vs Sales Lift ===')
print(discount_depth_df[['DISCOUNT_BAND', 'NUM_TRANSACTIONS', 'AVG_DISCOUNT_PCT', 
                          'QUANTITY_LIFT_VS_BASELINE_PCT', 'SALES_LIFT_VS_BASELINE_PCT', 
                          'NET_REVENUE_AFTER_DISCOUNT']].to_string(index=False))

print('\n=== Key Finding ===')
optimal = discount_depth_df[discount_depth_df['SALES_LIFT_VS_BASELINE_PCT'] == discount_depth_df['SALES_LIFT_VS_BASELINE_PCT'].max()]
print(f'Optimal Discount Band: {optimal["DISCOUNT_BAND"].values[0]}')
print(f'Max Sales Lift: {optimal["SALES_LIFT_VS_BASELINE_PCT"].values[0]:.1f}%')
print(f'\nDiminishing Returns: Beyond 20% discount, sales lift turns NEGATIVE')
print(f'Recommendation: Keep discounts in 1-10% range for maximum ROI')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = discount_depth_df['AVG_DISCOUNT_PCT'].values
y_lift = discount_depth_df['SALES_LIFT_VS_BASELINE_PCT'].values
y_net = discount_depth_df['NET_REVENUE_AFTER_DISCOUNT'].values

axes[0].plot(x, y_lift, 'o-', color='coral', linewidth=2, markersize=8)
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[0].fill_between(x, y_lift, 0, where=[l > 0 for l in y_lift], alpha=0.2, color='green', label='Positive Lift')
axes[0].fill_between(x, y_lift, 0, where=[l <= 0 for l in y_lift], alpha=0.2, color='red', label='Negative Lift')
axes[0].set_title('Sales Lift % vs Discount Depth')
axes[0].set_xlabel('Average Discount %')
axes[0].set_ylabel('Sales Lift vs Baseline %')
axes[0].legend()

axes[1].bar(discount_depth_df['DISCOUNT_BAND'], y_net, color=['#2ecc71' if n > 5 else '#f39c12' if n > 2 else '#e74c3c' for n in y_net], edgecolor='black')
axes[1].set_title('Net Revenue After Discount by Band')
axes[1].set_xlabel('Discount Band')
axes[1].set_ylabel('Net Revenue ($)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('/tmp/discount_depth_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Pantry Loading Detection

In [ ]:
%%sql -r pantry_df
SELECT * FROM MEGHA_PAWAR_COGNIZANT_COM_DB.PROMO_MARTS_PROMO_MARTS.MART_PANTRY_LOADING ORDER BY CAMPAIGN_ID

In [ ]:
print('=== Pantry Loading Analysis ===')
flag_counts = pantry_df['PANTRY_LOADING_FLAG'].value_counts()
print(flag_counts.to_string())

print(f'\n=== Campaigns with Pantry Loading ===')
pantry_campaigns = pantry_df[pantry_df['PANTRY_LOADING_FLAG'].str.contains('PANTRY')]
for _, row in pantry_campaigns.iterrows():
    print(f'Campaign {row["CAMPAIGN_ID"]} ({row["CAMPAIGN_TYPE"]}): '
          f'During lift={row["DURING_LIFT_PCT"]:.1f}%, Post dip={row["POST_CHANGE_PCT"]:.1f}%')

print(f'\n=== Key Finding ===')
print(f'{len(pantry_campaigns)} of {len(pantry_df)} campaigns ({len(pantry_campaigns)/len(pantry_df)*100:.0f}%) show pantry loading')
print('These campaigns did NOT generate real demand — customers just bought earlier')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_flag = ['#2ecc71' if 'TRUE' in f else '#e74c3c' for f in pantry_df['PANTRY_LOADING_FLAG']]
axes[0].bar(pantry_df['CAMPAIGN_ID'].astype(str), pantry_df['POST_CHANGE_PCT'], color=colors_flag, edgecolor='black')
axes[0].axhline(y=-15, color='red', linestyle='--', label='Pantry Loading Threshold (-15%)')
axes[0].set_title('Post-Campaign Sales Change % (Green=Good, Red=Pantry Loading)')
axes[0].set_xlabel('Campaign ID')
axes[0].set_ylabel('Post vs Pre Change %')
axes[0].legend()

spend_data = pantry_df[['CAMPAIGN_ID', 'AVG_PRE_SPEND', 'AVG_DURING_SPEND', 'AVG_POST_SPEND']].head(10)
bar_width = 0.25
x_pos = range(len(spend_data))
axes[1].bar([p - bar_width for p in x_pos], spend_data['AVG_PRE_SPEND'], bar_width, label='Pre', color='steelblue')
axes[1].bar(x_pos, spend_data['AVG_DURING_SPEND'], bar_width, label='During', color='coral')
axes[1].bar([p + bar_width for p in x_pos], spend_data['AVG_POST_SPEND'], bar_width, label='Post', color='#2ecc71')
axes[1].set_title('Pre/During/Post Spend (First 10 Campaigns)')
axes[1].set_xlabel('Campaign ID')
axes[1].set_ylabel('Avg Spend ($)')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(spend_data['CAMPAIGN_ID'].astype(str))
axes[1].legend()

plt.tight_layout()
plt.savefig('/tmp/pantry_loading.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Halo Effect Analysis

In [ ]:
%%sql -r halo_df
SELECT * FROM MEGHA_PAWAR_COGNIZANT_COM_DB.PROMO_MARTS_PROMO_MARTS.MART_HALO_EFFECT

In [ ]:
print('=== Halo Effect: Do Promo Items Drive Extra Purchases? ===')
print(halo_df.to_string(index=False))

promo_basket = halo_df[halo_df['BASKET_TYPE'] == 'Contains Promo Item']
no_promo_basket = halo_df[halo_df['BASKET_TYPE'] == 'No Promo Item']

if len(promo_basket) > 0 and len(no_promo_basket) > 0:
    size_diff = promo_basket['AVG_BASKET_SIZE'].values[0] - no_promo_basket['AVG_BASKET_SIZE'].values[0]
    val_diff = promo_basket['AVG_BASKET_VALUE'].values[0] - no_promo_basket['AVG_BASKET_VALUE'].values[0]
    print(f'\nBasket Size Difference: {size_diff:+.2f} items')
    print(f'Basket Value Difference: ${val_diff:+.2f}')
    print(f'\nConclusion: {"Halo effect present — promo baskets are larger" if size_diff > 0.1 else "Minimal halo effect in this dataset"}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

types = halo_df['BASKET_TYPE'].values
colors = ['coral', 'steelblue']

axes[0].bar(types, halo_df['AVG_BASKET_SIZE'], color=colors, edgecolor='black')
axes[0].set_title('Avg Basket Size by Type')
axes[0].set_ylabel('Avg Items per Basket')
for i, v in enumerate(halo_df['AVG_BASKET_SIZE']):
    axes[0].text(i, v + 0.01, f'{v:.2f}', ha='center', fontweight='bold')

axes[1].bar(types, halo_df['AVG_BASKET_VALUE'], color=colors, edgecolor='black')
axes[1].set_title('Avg Basket Value by Type')
axes[1].set_ylabel('Avg Value ($)')
for i, v in enumerate(halo_df['AVG_BASKET_VALUE']):
    axes[1].text(i, v + 0.05, f'${v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/halo_effect.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Promo Response by Customer Segment

In [ ]:
%%sql -r segment_promo_df
SELECT * FROM MEGHA_PAWAR_COGNIZANT_COM_DB.PROMO_MARTS_PROMO_MARTS.MART_SEGMENT_PROMO_RESPONSE

In [ ]:
print('=== Promo Effectiveness by Customer Segment ===')
print(segment_promo_df[['CUSTOMER_SEGMENT', 'NUM_HOUSEHOLDS', 'AVG_MONETARY', 
                         'PROMO_SPEND_PCT', 'PROMO_LIFT_BY_SEGMENT_PCT', 
                         'AVG_DISCOUNT_TAKEN']].to_string(index=False))

print(f'\n=== Key Findings ===')
best_seg = segment_promo_df.loc[segment_promo_df['PROMO_LIFT_BY_SEGMENT_PCT'].idxmax()]
worst_seg = segment_promo_df.loc[segment_promo_df['PROMO_LIFT_BY_SEGMENT_PCT'].idxmin()]
print(f'Most promo-responsive segment: {best_seg["CUSTOMER_SEGMENT"]} ({best_seg["PROMO_LIFT_BY_SEGMENT_PCT"]:+.2f}% lift)')
print(f'Least promo-responsive segment: {worst_seg["CUSTOMER_SEGMENT"]} ({worst_seg["PROMO_LIFT_BY_SEGMENT_PCT"]:+.2f}% lift)')
print(f'\nRecommendation: Target promos at "{best_seg["CUSTOMER_SEGMENT"]}" for maximum impact')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

segs = segment_promo_df['CUSTOMER_SEGMENT'].values
colors_seg = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']

lifts = segment_promo_df['PROMO_LIFT_BY_SEGMENT_PCT'].values
bar_colors = ['#2ecc71' if l > 0 else '#e74c3c' for l in lifts]
axes[0].barh(segs, lifts, color=bar_colors, edgecolor='black')
axes[0].axvline(x=0, color='black', linestyle='--')
axes[0].set_title('Promo Lift % by Customer Segment')
axes[0].set_xlabel('Promo Lift %')

axes[1].bar(segs, segment_promo_df['AVG_MONETARY'], color=colors_seg, edgecolor='black')
axes[1].set_title('Avg Lifetime Spend by Segment')
axes[1].set_ylabel('Avg Monetary ($)')
axes[1].tick_params(axis='x', rotation=30)

axes[2].bar(segs, segment_promo_df['NUM_HOUSEHOLDS'], color=colors_seg, edgecolor='black')
axes[2].set_title('Segment Size (# Households)')
axes[2].set_ylabel('Number of Households')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('/tmp/segment_promo_response.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Cannibalization Analysis

In [ ]:
%%sql -r cannibal_df
SELECT * FROM MEGHA_PAWAR_COGNIZANT_COM_DB.PROMO_MARTS_PROMO_MARTS.MART_CANNIBALIZATION ORDER BY CANNIBALIZATION_PCT ASC LIMIT 20

In [ ]:
print('=== Cannibalization Analysis ===')
print(f'Total categories analyzed: {len(cannibal_df)}')

level_counts = cannibal_df['CANNIBALIZATION_LEVEL'].value_counts()
print(f'\nCannibalization Levels:')
print(level_counts.to_string())

print(f'\n=== Top Cannibalized Categories ===')
high_cannibal = cannibal_df[cannibal_df['CANNIBALIZATION_LEVEL'] == 'HIGH CANNIBALIZATION'].head(5)
if len(high_cannibal) > 0:
    print(high_cannibal[['DEPARTMENT', 'COMMODITY_DESC', 'CANNIBALIZATION_PCT', 'CANNIBALIZATION_LEVEL']].to_string(index=False))
    print(f'\nRecommendation: Avoid heavy promotions in these categories — they cannibalize other products')
else:
    print('No high cannibalization detected in top categories')
    moderate = cannibal_df[cannibal_df['CANNIBALIZATION_LEVEL'] == 'MODERATE CANNIBALIZATION'].head(5)
    if len(moderate) > 0:
        print(f'\nModerate cannibalization found:')
        print(moderate[['DEPARTMENT', 'COMMODITY_DESC', 'CANNIBALIZATION_PCT']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

top_categories = cannibal_df.head(15)
colors_can = ['#e74c3c' if l == 'HIGH CANNIBALIZATION' else '#f39c12' if l == 'MODERATE CANNIBALIZATION' else '#2ecc71' 
              for l in top_categories['CANNIBALIZATION_LEVEL']]

ax.barh(top_categories['COMMODITY_DESC'].str[:30], top_categories['CANNIBALIZATION_PCT'].fillna(0), 
        color=colors_can, edgecolor='black')
ax.axvline(x=-10, color='red', linestyle='--', label='High Cannibalization Threshold')
ax.axvline(x=-5, color='orange', linestyle='--', label='Moderate Threshold')
ax.set_title('Cannibalization % by Product Category')
ax.set_xlabel('Cannibalization % (more negative = worse)')
ax.legend()

plt.tight_layout()
plt.savefig('/tmp/cannibalization.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 13. FINAL EXECUTIVE SUMMARY

In [ ]:
print('=' * 70)
print('   PRICING & PROMOTION EFFECTIVENESS - FINAL EXECUTIVE SUMMARY')
print('=' * 70)

print(f'\n{"="*70}')
print('   1. DISCOUNT DEPTH OPTIMIZATION')
print(f'{"="*70}')
print(f'   - Optimal discount: 1-10% (delivers 31-36% sales lift)')
print(f'   - Beyond 20% discount: NEGATIVE returns (margin destroyed)')
print(f'   - Sweet spot: Max ROI at 1-5% band with $10.17 net revenue')

print(f'\n{"="*70}')
print('   2. PANTRY LOADING DETECTION')
print(f'{"="*70}')
pantry_count = len(pantry_df[pantry_df['PANTRY_LOADING_FLAG'].str.contains('PANTRY')])
print(f'   - {pantry_count} of 30 campaigns ({pantry_count/30*100:.0f}%) show pantry loading')
print(f'   - These campaigns gave discounts but did NOT increase total demand')
print(f'   - Campaigns 15, 20, 23, 24, 25 should be discontinued or redesigned')

print(f'\n{"="*70}')
print('   3. HALO EFFECT')
print(f'{"="*70}')
print(f'   - Baskets with promo items: {promo_basket["NUM_BASKETS"].values[0]:,} baskets')
print(f'   - Minimal halo in this dataset (single-item baskets dominate)')
print(f'   - Real-world implication: bundle promotions for cross-sell')

print(f'\n{"="*70}')
print('   4. CUSTOMER SEGMENT TARGETING')
print(f'{"="*70}')
for _, row in segment_promo_df.iterrows():
    print(f'   - {row["CUSTOMER_SEGMENT"]}: {row["NUM_HOUSEHOLDS"]} HHs, '
          f'Promo Lift={row["PROMO_LIFT_BY_SEGMENT_PCT"]:+.2f}%, '
          f'Avg Spend=${row["AVG_MONETARY"]:,.0f}')

print(f'\n{"="*70}')
print('   5. CANNIBALIZATION')
print(f'{"="*70}')
high_can = cannibal_df[cannibal_df['CANNIBALIZATION_LEVEL'] == 'HIGH CANNIBALIZATION']
mod_can = cannibal_df[cannibal_df['CANNIBALIZATION_LEVEL'] == 'MODERATE CANNIBALIZATION']
print(f'   - High cannibalization categories: {len(high_can)}')
print(f'   - Moderate cannibalization categories: {len(mod_can)}')
print(f'   - These categories lose other-product sales when one product is promoted')

print(f'\n{"="*70}')
print('   TOP 5 RECOMMENDATIONS')
print('=' * 70)
print('   1. Cap all discounts at 15% maximum — beyond this, ROI is negative')
print('   2. Kill campaigns 15, 20, 23, 24, 25 — pure pantry loading')
print('   3. Invest more in display promos (higher lift than mailers)')
print('   4. Target Champions & At-Risk segments differently:')
print('      - Champions: loyalty rewards (they buy anyway)')
print('      - At Risk: retention offers (bring them back)')
print('   5. Avoid deep promotions in high-cannibalization categories')
print(f'\n{"="*70}')